<a href="https://colab.research.google.com/github/Zubeyr2k/Preliminary-work/blob/main/TweetDome_Preliminary_Work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TweetDome: Preliminary Work (EDA and baselines)



## Set up

In [ ]:
import os, glob, re, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/TweetDome/data'
except ImportError:                      # running outside Colab
    DATA_DIR = os.environ.get('TD_DATA', '.')
FIG_DIR = os.path.join(os.path.dirname(DATA_DIR.rstrip('/')), 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# clear labels, no chartjunk.
sns.set_theme(style='whitegrid', palette='colorblind', context='notebook')
TRUE_FALSE = {'True': '#0072B2', 'False': '#E69F00'}   # blue-orange, colour-blind safe
def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=300, bbox_inches='tight')
    print('saved', name)
RANDOM_STATE = 42

## Load, clean and anonymise


In [ ]:
ts_path = glob.glob(os.path.join(DATA_DIR, 'Truth_Seeker_Model_Dataset_With_TimeStamps*.xlsx'))[0]
feat_path = glob.glob(os.path.join(DATA_DIR, 'Features_For_Traditional_ML_Techniques*.csv'))[0]

raw = pd.read_excel(ts_path)
print('Raw rows:', len(raw))
raw['dt'] = pd.to_datetime(raw['timestamp'], format='%a %b %d %H:%M:%S %z %Y', errors='coerce')

log = {'raw rows': len(raw)}
broken = raw['dt'].isna() | raw['tweet'].isna() | raw['BinaryNumTarget'].isna()
df = raw.loc[~broken].copy()
log['broken or missing rows removed'] = int(broken.sum())

n = len(df); df = df.drop_duplicates(subset='tweet')
log['exact duplicate tweets removed'] = n - len(df)

df['label'] = np.where(df['3_label_majority_answer'] == 'Agree',
                       df['BinaryNumTarget'], 1 - df['BinaryNumTarget']).astype(int)   # 1 = true tweet

def mask(text):
    text = re.sub(r'https?://\S+', '[URL]', str(text))
    return re.sub(r'@\w+', '[USER]', text)
df['tweet_clean'] = df['tweet'].map(mask)

df = df.sort_values('dt').reset_index(drop=True)
log['clean rows'] = len(df)
cleaning_log = pd.Series(log, name='rows'); print(cleaning_log)
cleaning_log.to_csv(os.path.join(FIG_DIR, 'cleaning_log.csv'))

In [ ]:
feat = pd.read_csv(feat_path)
ACCOUNT = ['followers_count', 'friends_count', 'favourites_count', 'statuses_count',
           'listed_count', 'BotScore', 'cred', 'normalize_influence']
feat = feat.drop_duplicates(subset='tweet')[['tweet', 'majority_target'] + ACCOUNT]
df = df.merge(feat, on='tweet', how='left')
print('Tweets without account features:', df['cred'].isna().sum())
# Validation check: rebuilt labels should match the dataset's own tweet label
ok = df.dropna(subset=['majority_target'])
print('Label agreement with majority_target:', (ok['label'] == ok['majority_target'].astype(int)).mean().round(4))
df = df.dropna(subset=ACCOUNT).reset_index(drop=True)

## Step 2: Exploratory data analysis


In [ ]:
print('Date range:', df['dt'].min().date(), 'to', df['dt'].max().date())
print('Statements:', df['statement'].nunique(), '| Tweets:', len(df))
print(df['label'].map({1: 'True', 0: 'False'}).value_counts(normalize=True).round(3))
df['words'] = df['tweet_clean'].str.split().str.len()
df[['words'] + ACCOUNT].describe().round(2)

In [ ]:
# Histogram of tweet length by label.
fig, ax = plt.subplots(figsize=(7, 3.5))
lab = df['label'].map({1: 'True', 0: 'False'})
sns.histplot(data=df.assign(Label=lab), x='words', hue='Label', binwidth=5, element='step',
             stat='density', common_norm=False, palette=TRUE_FALSE, ax=ax)
ax.set(title='Tweet length by label', xlabel='Words per tweet', ylabel='Density')
save(fig, 'eda_tweet_length.png'); plt.show()

In [ ]:
# Box plots of account features by label. Counts are heavily skewed, so a log scale will need to be used.
# BotScore is left out: its median is 0.03 for both labels, so its box plot is a flat line.
plot_cols = ['followers_count', 'statuses_count', 'cred', 'normalize_influence']
long = df.assign(Label=lab).melt(id_vars='Label', value_vars=plot_cols)
g = sns.catplot(data=long, x='Label', y='value', col='variable', kind='box', sharey=False,
                palette=TRUE_FALSE, height=3, aspect=0.8, showfliers=False)
g.axes.flat[0].set_yscale('log'); g.axes.flat[1].set_yscale('log')
g.set_titles('{col_name}'); g.set_axis_labels('', '')
save(g.fig, 'eda_account_features.png'); plt.show()

In [ ]:
# Correlogram (heatmap of a correlation matrix) with a diverging palette centred on zero.
# Spearman, because the counts are skewed and the relationships may be monotonic, not linear.
corr = df[ACCOUNT + ['label']].corr(method='spearman')
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f',
            annot_kws={'size': 7}, ax=ax)
ax.set_title("Spearman correlation of account features and tweet label")
save(fig, 'eda_correlogram.png'); plt.show()

## Step 3: Chronological windows and leakage audit


In [ ]:
n = len(df)
edges = [0, int(n * .5), int(n * .625), int(n * .75), int(n * .875), n]
df['window'] = 0
for i in range(5):
    df.loc[edges[i]:edges[i + 1] - 1, 'window'] = i

seen = set(df.loc[df.window == 0, 'statement'])
rows = []
for i in range(5):
    w = df[df.window == i]
    rows.append({'window': f'W{i}', 'start': w.dt.min().date(), 'end': w.dt.max().date(),
                 'tweets': len(w), '% true': round(w.label.mean() * 100, 1),
                 'statements': w.statement.nunique(),
                 '% tweets on statements seen in W0': None if i == 0 else
                     round(w.statement.isin(seen).mean() * 100, 1)})
windows = pd.DataFrame(rows); windows.to_csv(os.path.join(FIG_DIR, 'windows.csv'), index=False)
windows

In [ ]:
# Time series of tweets per month with the windows shaded and labelled.
monthly = df.set_index('dt').resample('MS').size()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(monthly.index, monthly.values, color='#333333', lw=1)
shades = ['#0072B2', '#56B4E9', '#009E73', '#E69F00', '#CC79A7']
from matplotlib.patches import Patch
handles = []
for i in range(5):
    w = df[df.window == i]
    ax.axvspan(w.dt.min(), w.dt.max(), color=shades[i], alpha=0.25)
    role = 'train' if i == 0 else 'test'
    handles.append(Patch(color=shades[i], alpha=0.5,
                         label=f"W{i} {role}: {w.dt.min():%b %Y} to {w.dt.max():%b %Y}"))
ax.legend(handles=handles, fontsize=8, loc='upper left', title='Equal-count windows', title_fontsize=8)
ax.set(title='TruthSeeker tweets per month', xlabel='Month', ylabel='Tweets')
save(fig, 'windows_timeline.png'); plt.show()

In [ ]:
# Proportions per window: a 100% stacked bar shows how the true/false mix changes over time.
prop = pd.crosstab(df['window'].map(lambda i: f'W{i}'), lab, normalize='index')[['True', 'False']]
fig, ax = plt.subplots(figsize=(6, 3.2))
prop.plot(kind='bar', stacked=True, color=[TRUE_FALSE['True'], TRUE_FALSE['False']], ax=ax, rot=0)
for c in ax.containers:
    ax.bar_label(c, labels=[f'{v*100:.0f}%' for v in c.datavalues], label_type='center', fontsize=8, color='white')
ax.set(title='Share of true and false tweets per window', xlabel='', ylabel='Proportion')
ax.legend(title='', bbox_to_anchor=(1, 1))
save(fig, 'windows_class_balance.png'); plt.show()

## Step 4: Baseline models


In [ ]:
w0 = df[df.window == 0]
cut = int(len(w0) * 0.9)
train, val = w0.iloc[:cut], w0.iloc[cut:]
tests = {f'W{i}': df[df.window == i] for i in range(1, 5)}

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000, sublinear_tf=True)
Xtr_t = tfidf.fit_transform(train.tweet_clean); Xva_t = tfidf.transform(val.tweet_clean)
ytr, yva = train.label.values, val.label.values
mf1 = lambda y, p: f1_score(y, p, average='macro')

In [ ]:
# Tune each model on the validation set only.
best = {}
best['Majority class'] = DummyClassifier(strategy='most_frequent').fit(Xtr_t, ytr)

scores = {C: mf1(yva, LogisticRegression(C=C, max_iter=2000).fit(Xtr_t, ytr).predict(Xva_t)) for C in [0.1, 1, 10]}
C = max(scores, key=scores.get); print('LogReg C:', C, scores)
best['TF-IDF + Logistic Regression'] = LogisticRegression(C=C, max_iter=2000).fit(Xtr_t, ytr)

# kNN is a lazy learner: prediction time grows with training size, so a random 20,000-tweet sample is used.
idx = np.random.RandomState(RANDOM_STATE).choice(Xtr_t.shape[0], min(20000, Xtr_t.shape[0]), replace=False)
scores = {k: mf1(yva, KNeighborsClassifier(k, metric='cosine', algorithm='brute').fit(Xtr_t[idx], ytr[idx]).predict(Xva_t)) for k in [5, 15, 31]}
k = max(scores, key=scores.get); print('kNN k:', k, scores)
best['TF-IDF + kNN'] = KNeighborsClassifier(k, metric='cosine', algorithm='brute').fit(Xtr_t[idx], ytr[idx])

In [ ]:
# Random forest on account features only. Comparing train vs validation macro-F1 at each depth
# to see under/overfitting.
Xtr_a, Xva_a = train[ACCOUNT].values, val[ACCOUNT].values
fit_rows = []
for d in [3, 6, 12, None]:
    rf = RandomForestClassifier(n_estimators=300, max_depth=d, oob_score=True,
                                n_jobs=-1, random_state=RANDOM_STATE).fit(Xtr_a, ytr)
    fit_rows.append({'max_depth': str(d), 'train macro-F1': mf1(ytr, rf.predict(Xtr_a)),
                     'validation macro-F1': mf1(yva, rf.predict(Xva_a)), 'OOB accuracy': rf.oob_score_})
fit_table = pd.DataFrame(fit_rows).round(3); print(fit_table)
d = fit_rows[int(np.argmax([r['validation macro-F1'] for r in fit_rows]))]['max_depth']
best['Random Forest (account features)'] = RandomForestClassifier(
    n_estimators=300, max_depth=None if d == 'None' else int(d), n_jobs=-1,
    random_state=RANDOM_STATE).fit(Xtr_a, ytr)
fit_table.to_csv(os.path.join(FIG_DIR, 'rf_fit.csv'), index=False)

In [ ]:
# Evaluate every baseline on each later window
def features(name, d):
    return d[ACCOUNT].values if 'Random Forest' in name else tfidf.transform(d.tweet_clean)

res = []
for name, model in best.items():
    for w, d in tests.items():
        X, y = features(name, d), d.label.values
        p = model.predict(X); prob = model.predict_proba(X)[:, 1]
        res.append({'model': name, 'window': w, 'accuracy': accuracy_score(y, p),
                    'macro-F1': mf1(y, p), 'ROC-AUC': roc_auc_score(y, prob)})
results = pd.DataFrame(res).round(3)
results.to_csv(os.path.join(FIG_DIR, 'baseline_results.csv'), index=False)
results.pivot(index='model', columns='window', values='macro-F1')

In [ ]:
# Macro-F1 over time for each static baseline.
fig, ax = plt.subplots(figsize=(7, 3.8))
markers = ['o', 's', '^', 'D']
for (name, g), m in zip(results.groupby('model', sort=False), markers):
    ax.plot(g['window'], g['macro-F1'], marker=m, label=name)
ax.set(title='Static baselines: macro-F1 on later windows', xlabel='Test window', ylabel='Macro-F1', ylim=(0, 1))
ax.legend(fontsize=8, loc='lower left')
save(fig, 'baselines_macro_f1.png'); plt.show()

In [ ]:
# Confusion matrix for the best text baseline on the first test window (row-normalised).
name = 'TF-IDF + Logistic Regression'; d = tests['W1']
cm = confusion_matrix(d.label, best[name].predict(features(name, d)), normalize='true')
fig, ax = plt.subplots(figsize=(3.8, 3.4))
ConfusionMatrixDisplay(cm, display_labels=['False', 'True']).plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
ax.set_title(f'{name} on W1')
save(fig, 'confusion_w1.png'); plt.show()

In [ ]:
# Leakage check: is the model better on tweets about statements it saw in training?
rows = []
for w, d in tests.items():
    for flag, part in d.groupby(d.statement.isin(seen)):
        p = best['TF-IDF + Logistic Regression'].predict(tfidf.transform(part.tweet_clean))
        rows.append({'window': w, 'statement seen in W0': flag, 'tweets': len(part), 'macro-F1': mf1(part.label, p)})
seen_table = pd.DataFrame(rows).round(3); seen_table.to_csv(os.path.join(FIG_DIR, 'seen_vs_unseen.csv'), index=False)
seen_table

## Step 5: Significance tests


In [ ]:
chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(df.window, df.label))
print(f'Label vs window: chi-square = {chi2:.1f}, df = {dof}, p = {p:.3g}')
u, p = stats.mannwhitneyu(df.loc[df.label == 1, 'cred'], df.loc[df.label == 0, 'cred'])
print(f'Credibility, true vs false tweets: Mann-Whitney U = {u:.3g}, p = {p:.3g}')
print('Median cred (true, false):', df.loc[df.label == 1, 'cred'].median(), df.loc[df.label == 0, 'cred'].median())